# FEATURE ENGINEERING

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## Загрузка данных

In [2]:
application_train = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/application_train.csv")
application_test = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/application_test.csv")
bureau = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/bureau.csv")
installments = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/installments_payments.csv")
previous = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/previous_application.csv")
pos = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/POS_CASH_balance.csv")
cc = pd.read_csv("/kaggle/input/competitions/home-credit-default-risk/credit_card_balance.csv")

## Признаки для датасета application

In [3]:
building = [
    'APARTMENTS_AVG',
    'BASEMENTAREA_AVG',
    'YEARS_BEGINEXPLUATATION_AVG',
    'YEARS_BUILD_AVG',
    'COMMONAREA_AVG',
    'ELEVATORS_AVG',
    'ENTRANCES_AVG',
    'FLOORSMAX_AVG',
    'FLOORSMIN_AVG',
    'LANDAREA_AVG',
    'LIVINGAPARTMENTS_AVG',
    'LIVINGAREA_AVG',
    'NONLIVINGAPARTMENTS_AVG',
    'NONLIVINGAREA_AVG',

    'APARTMENTS_MODE',
    'BASEMENTAREA_MODE',
    'YEARS_BEGINEXPLUATATION_MODE',
    'YEARS_BUILD_MODE',
    'COMMONAREA_MODE',
    'ELEVATORS_MODE',
    'ENTRANCES_MODE',
    'FLOORSMAX_MODE',
    'FLOORSMIN_MODE',
    'LANDAREA_MODE',
    'LIVINGAPARTMENTS_MODE',
    'LIVINGAREA_MODE',
    'NONLIVINGAPARTMENTS_MODE',
    'NONLIVINGAREA_MODE',

    'APARTMENTS_MEDI',
    'BASEMENTAREA_MEDI',
    'YEARS_BEGINEXPLUATATION_MEDI',
    'YEARS_BUILD_MEDI',
    'COMMONAREA_MEDI',
    'ELEVATORS_MEDI',
    'ENTRANCES_MEDI',
    'FLOORSMAX_MEDI',
    'FLOORSMIN_MEDI',
    'LANDAREA_MEDI',
    'LIVINGAPARTMENTS_MEDI',
    'LIVINGAREA_MEDI',
    'NONLIVINGAPARTMENTS_MEDI',
    'NONLIVINGAREA_MEDI',

    'FONDKAPREMONT_MODE',
    'HOUSETYPE_MODE',
    'TOTALAREA_MODE',
    'WALLSMATERIAL_MODE',
    'EMERGENCYSTATE_MODE'
]

На основе характеристик заявителей создаются дополнительные признаки, отражающие возраст, кредитную нагрузку, финансовое положение и другие особенности клиентов.

Основные группы новых признаков:
- Финансовые отношения: отношение дохода к ежемесячному платежу, суммы кредита к доходу, кредита к стоимости приобретаемого товара и другие показатели.
- Возраст и трудоустройство: возраст заёмщика и отношения между временными характеристиками.
- Внешние оценки: среднее, минимум, максимум, дисперсия и другие агрегаты EXT_SOURCE_1, EXT_SOURCE_2, EXT_SOURCE_3.
- Документы: количество предоставленных документов и эксцесс бинарных признаков документов.
- Пропуски: количество отсутствующих значений среди характеристик недвижимости.

Для предотвращения деления на ноль нулевые знаменатели заменяются на NaN.

Все преобразования выполняются одинаково для обучающей и тестовой выборок.

In [4]:
def application_features(df):
    df_new = df.copy()
    df_new["age"] = df["DAYS_BIRTH"] / (-365)
    df_new["income_credit_relation"] = df["AMT_INCOME_TOTAL"] / df["AMT_ANNUITY"].replace(0, np.nan)
    df_new["BUILDING_MISSING_COUNT"] = df[building].isna().sum(axis=1)
    df_new["credit_annuity_relation"] = df["AMT_CREDIT"] / df["AMT_ANNUITY"].replace(0, np.nan)
    df_new["single_young_man"] = (df["CODE_GENDER"] == "M") & (df["NAME_FAMILY_STATUS"] != "Married") & (df_new["age"] < 40)
    df_new["goods_price_to_income"] = df["AMT_INCOME_TOTAL"] / df["AMT_GOODS_PRICE"].replace(0, np.nan)

    docs = [col for col in df.columns if "FLAG_DOC" in col]
    
    df_new["DOCUMENT_COUNT"] = df[docs].sum(axis=1)
    df_new["NEW_DOC_KURT"] = df[docs].kurtosis(axis=1)
    
    df_new["AGE_RANGE"] = pd.cut(
        df_new["age"],
        bins=[0, 27, 40, 50, 65, 99, np.inf],
        labels=[1, 2, 3, 4, 5, 0],
        right=False
    ).astype(int)
    
    ext = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
    
    df_new["EXT_SOURCES_PROD"] = ext.prod(axis=1, min_count=3)
    df_new["EXT_SOURCES_WEIGHTED"] = (
        df["EXT_SOURCE_1"] * 2
        + df["EXT_SOURCE_2"]
        + df["EXT_SOURCE_3"] * 3
    )
    
    df_new["EXT_SOURCES_MIN"] = ext.min(axis=1)
    df_new["EXT_SOURCES_MAX"] = ext.max(axis=1)
    df_new["EXT_SOURCES_MEAN"] = ext.mean(axis=1)
    df_new["EXT_SOURCES_NANMEDIAN"] = ext.median(axis=1)
    df_new["EXT_SOURCES_VAR"] = ext.var(axis=1, ddof=0)
    df_new["CREDIT_TO_GOODS_RATIO"] = (
        df["AMT_CREDIT"] / df["AMT_GOODS_PRICE"].replace(0, np.nan)
    )
    
    df_new["ANNUITY_TO_INCOME_RATIO"] = (
        df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )
    
    df_new["CREDIT_TO_INCOME_RATIO"] = (
        df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )
    
    df_new["INCOME_TO_EMPLOYED_RATIO"] = (
        df["AMT_INCOME_TOTAL"] / df["DAYS_EMPLOYED"].replace(365243, np.nan).replace(0, np.nan)
    )
    
    df_new["INCOME_TO_BIRTH_RATIO"] = (
        df["AMT_INCOME_TOTAL"] / df["DAYS_BIRTH"].replace(0, np.nan)
    )
    df_new["EMPLOYED_TO_BIRTH_RATIO"] = (
        df["DAYS_EMPLOYED"].replace(365243, np.nan) / df["DAYS_BIRTH"].replace(0, np.nan)
    )
    
    df_new["ID_TO_BIRTH_RATIO"] = (
        df["DAYS_ID_PUBLISH"] / df["DAYS_BIRTH"].replace(0, np.nan)
    )
    
    df_new["CAR_TO_BIRTH_RATIO"] = (
        df["OWN_CAR_AGE"] / df["DAYS_BIRTH"].replace(0, np.nan)
    )
    
    df_new["CAR_TO_EMPLOYED_RATIO"] = (
        df["OWN_CAR_AGE"] / df["DAYS_EMPLOYED"].replace(365243, np.nan)
    )
    
    df_new["PHONE_TO_BIRTH_RATIO"] = (
        df["DAYS_LAST_PHONE_CHANGE"].replace(0, np.nan)
        / df["DAYS_BIRTH"].replace(0, np.nan)
    )
    return df_new

## Bureau features

Таблица bureau содержит информацию о кредитах клиентов в других финансовых организациях.

На её основе создаются признаки, характеризующие кредитную историю и долговую нагрузку заёмщиков:

- Общее количество кредитов и количество действующих кредитов.

- Суммы задолженности, кредитов и просроченных платежей.

- Максимальная продолжительность просрочки.

- Средняя продолжительность кредитов.

- Отношение задолженности к сумме кредита.

- Дополнительно рассчитываются характеристики действующих кредитов и кредитов, открытых за последние 365 дней.

Информация агрегируется по SK_ID_CURR, чтобы каждому клиенту соответствовала одна строка.

In [5]:
bureau["CREDIT_DURATION"] = (
    bureau["DAYS_CREDIT_ENDDATE"] - bureau["DAYS_CREDIT"]
)

bureau["DEBT_CREDIT_RATIO"] = (
    bureau["AMT_CREDIT_SUM_DEBT"]
    / bureau["AMT_CREDIT_SUM"].replace(0, np.nan)
)

In [6]:
bureau_features = bureau.groupby("SK_ID_CURR").agg(
    CREDIT_COUNT=("SK_ID_BUREAU", "count"),
    ACTIVE_CREDIT_COUNT=("CREDIT_ACTIVE", lambda x: (x == "Active").sum()),
    TOTAL_DEBT=("AMT_CREDIT_SUM_DEBT", lambda x: x.sum(min_count=1)),
    MAX_OVERDUE=("AMT_CREDIT_MAX_OVERDUE", "max"),
    TOTAL_OVERDUE=("AMT_CREDIT_SUM_OVERDUE", lambda x: x.sum(min_count=1)),
    MAX_DAYS_OVERDUE=("CREDIT_DAY_OVERDUE", "max"),

    TOTAL_CREDIT=("AMT_CREDIT_SUM", lambda x: x.sum(min_count=1)),
    MEAN_CREDIT=("AMT_CREDIT_SUM", "mean"),
    MEAN_DEBT=("AMT_CREDIT_SUM_DEBT", "mean"),

    MEAN_CREDIT_DURATION=("CREDIT_DURATION", "mean"),
    MEAN_DEBT_CREDIT_RATIO=("DEBT_CREDIT_RATIO", "mean"),

    MEAN_CREDIT_AGE=("DAYS_CREDIT", "mean"),
    LAST_CREDIT_DAYS=("DAYS_CREDIT", "max"),
    CONSUMER_CREDIT_COUNT=("CREDIT_TYPE", lambda x: (x == "Consumer credit").sum()),
    CREDIT_CARD_COUNT=("CREDIT_TYPE", lambda x: (x == "Credit card").sum()),
    CAR_LOAN_COUNT=("CREDIT_TYPE", lambda x: (x == "Car loan").sum()),
    MORTGAGE_COUNT=("CREDIT_TYPE", lambda x: (x == "Mortgage").sum()),
    MICROLOAN_COUNT=("CREDIT_TYPE", lambda x: (x == "Microloan").sum()),

    CLOSED_CREDIT_COUNT=("CREDIT_ACTIVE", lambda x: (x == "Closed").sum()),
).reset_index()

In [7]:
active = bureau[bureau["CREDIT_ACTIVE"] == "Active"]

active_features = active.groupby("SK_ID_CURR").agg(
    ACTIVE_TOTAL_DEBT=("AMT_CREDIT_SUM_DEBT", lambda x: x.sum(min_count=1)),
    ACTIVE_TOTAL_CREDIT=("AMT_CREDIT_SUM", lambda x: x.sum(min_count=1)),
    ACTIVE_MEAN_DEBT=("AMT_CREDIT_SUM_DEBT", "mean"),
    ACTIVE_MAX_OVERDUE=("AMT_CREDIT_MAX_OVERDUE", "max"),
    ACTIVE_MEAN_CREDIT_DURATION=("CREDIT_DURATION", "mean"),
    ACTIVE_CONSUMER_CREDIT_COUNT=("CREDIT_TYPE", lambda x: (x == "Consumer credit").sum()),
    ACTIVE_CREDIT_CARD_COUNT=("CREDIT_TYPE", lambda x: (x == "Credit card").sum()),
    ACTIVE_CAR_LOAN_COUNT=("CREDIT_TYPE", lambda x: (x == "Car loan").sum()),
    ACTIVE_MORTGAGE_COUNT=("CREDIT_TYPE", lambda x: (x == "Mortgage").sum()),
    ACTIVE_MICROLOAN_COUNT=("CREDIT_TYPE", lambda x: (x == "Microloan").sum()),
).reset_index()

bureau_features = bureau_features.merge(
    active_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

In [8]:
bureau_features["ACTIVE_DEBT_CREDIT_RATIO"] = (
    bureau_features["ACTIVE_TOTAL_DEBT"]
    / bureau_features["ACTIVE_TOTAL_CREDIT"].replace(0, np.nan)
)

In [9]:
recent = bureau[bureau["DAYS_CREDIT"].between(-365, 0)]

recent_features = recent.groupby("SK_ID_CURR").agg(
    RECENT_CREDIT_COUNT=("SK_ID_BUREAU", "count"),
    RECENT_TOTAL_CREDIT=("AMT_CREDIT_SUM", lambda x: x.sum(min_count=1)),
    RECENT_MEAN_CREDIT=("AMT_CREDIT_SUM", "mean"),
    RECENT_TOTAL_DEBT=("AMT_CREDIT_SUM_DEBT", lambda x: x.sum(min_count=1)),
).reset_index()

bureau_features = bureau_features.merge(
    recent_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

In [10]:
print("Размер:", bureau_features.shape)
print("Дубликаты ID:", bureau_features["SK_ID_CURR"].duplicated().sum())
numeric = bureau_features.select_dtypes(include="number")
print("Количество inf:", np.isinf(numeric.to_numpy()).sum())

Размер: (305811, 35)
Дубликаты ID: 0
Количество inf: 0


## Installments features

Таблица installments_payments содержит историю запланированных и фактических платежей по предыдущим кредитам.

На её основе рассчитываются показатели платёжной дисциплины:

- DPD — количество дней просрочки.

- LATE_PAYMENT — индикатор просроченного платежа.

- UNDERPAYMENT — индикатор неполной оплаты.

- Разница между фактической и запланированной суммой платежа.

- Частота просрочек продолжительностью от 7, 15 и 30 дней.

Затем рассчитываются агрегаты на уровне клиента: количество платежей, доля просрочек, средняя и максимальная просрочка, суммы платежей и другие показатели.

Отдельно выделяются характеристики платежей за последние 365 дней.

При отсутствии фактической даты или суммы платежа неизвестный статус не приравнивается к отсутствию просрочки.

In [11]:
installments["DPD"] = (
    installments["DAYS_ENTRY_PAYMENT"]
    - installments["DAYS_INSTALMENT"]
).clip(lower=0)

installments["LATE_PAYMENT"] = (
    installments["DPD"] > 0
).where(installments["DPD"].notna()).astype(float)

installments["UNDERPAYMENT"] = (
    installments["AMT_PAYMENT"] < installments["AMT_INSTALMENT"]
).where(
    installments["AMT_PAYMENT"].notna()
    & installments["AMT_INSTALMENT"].notna()
).astype(float)

installments["AMT_PAYMENT_INSTALMENT_DIFF"] = (
    installments["AMT_PAYMENT"] - installments["AMT_INSTALMENT"]
)


In [12]:
installments_features = installments.groupby("SK_ID_CURR").agg(
    PAYMENT_COUNT=("SK_ID_PREV", "count"),

    LATE_PAYMENT_COUNT=("LATE_PAYMENT", "sum"),
    LATE_PAYMENT_RATE=("LATE_PAYMENT", "mean"),
    MAX_DPD=("DPD", "max"),
    MEAN_DPD=("DPD", "mean"),
    STD_DPD = ("DPD", "std"),

    UNDERPAYMENT_COUNT=("UNDERPAYMENT", "sum"),
    UNDERPAYMENT_RATE=("UNDERPAYMENT", "mean"),

    AMT_PAYMENT_INSTALMENT_DIFF_MEAN=("AMT_PAYMENT_INSTALMENT_DIFF", "mean"),
    AMT_PAYMENT_INSTALMENT_DIFF_STD = ("AMT_PAYMENT_INSTALMENT_DIFF", "std"),

    TOTAL_PAYMENT=("AMT_PAYMENT", "sum"),
    MEAN_PAYMENT=("AMT_PAYMENT", "mean"),
).reset_index()

In [13]:
installments["DPD_7"] = (installments["DPD"] >= 7).astype(int)
installments["DPD_15"] = (installments["DPD"] >= 15).astype(int)
installments["DPD_30"] = (installments["DPD"] >= 30).astype(int)

overdue_features = installments.groupby("SK_ID_CURR").agg(
    DPD_7_RATE=("DPD_7", "mean"),
    DPD_15_RATE=("DPD_15", "mean"),
    DPD_30_RATE=("DPD_30", "mean"),
).reset_index()

installments_features = installments_features.merge(
    overdue_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

In [14]:
recent_installments = installments[
    installments["DAYS_INSTALMENT"].between(-365, 0)
]

recent_features = recent_installments.groupby("SK_ID_CURR").agg(
    RECENT_PAYMENT_COUNT=("SK_ID_PREV", "count"),
    RECENT_LATE_PAYMENT_RATE=("LATE_PAYMENT", "mean"),
    RECENT_MAX_DPD=("DPD", "max"),
    RECENT_MEAN_DPD=("DPD", "mean"),
).reset_index()

installments_features = installments_features.merge(
    recent_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

In [15]:
print("Размер:", installments_features.shape)
print("Дубликаты ID:", installments_features["SK_ID_CURR"].duplicated().sum())
numeric = installments_features.select_dtypes(include="number")
print("Количество inf:", np.isinf(numeric.to_numpy()).sum())

Размер: (339587, 20)
Дубликаты ID: 0
Количество inf: 0


## Previous features

Таблица previous_application содержит информацию о предыдущих кредитных заявках клиентов.

Для характеристики кредитной активности рассчитываются:

- Количество предыдущих заявок.

- Средние и максимальные суммы запрашиваемых и одобренных кредитов.

- Отношения между запрашиваемой суммой, суммой кредита и ежемесячным платежом.

- Количество и доля одобренных и отклонённых заявок.

- Временные интервалы между предыдущими заявками.

- Количество завершённых кредитных договоров.

Дополнительно создаются агрегаты по одобренным заявкам и признаки, характеризующие типы кредитов, причины отказов и цели кредитования.

Для оценки недавней кредитной активности отдельно рассматриваются заявки за последние 90, 180 и 365 дней.

Значение 365243 в столбцах дат заменяется на NaN, поскольку используется как специальное обозначение отсутствующей или неприменимой даты.

Все результаты агрегируются до уровня одного клиента (SK_ID_CURR).

In [16]:
days_cols = [
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION"
]

previous[days_cols] = previous[days_cols].replace(365243, np.nan)

previous["APPLICATION_CREDIT_RATIO"] = (
    previous["AMT_CREDIT"]
    / previous["AMT_APPLICATION"].replace(0, np.nan)
)

previous["APPLICATION_CREDIT_DIFF"] = (
    previous["AMT_APPLICATION"] - previous["AMT_CREDIT"]
)

previous["DOWN_PAYMENT_RATIO"] = (
    previous["AMT_DOWN_PAYMENT"]
    / previous["AMT_CREDIT"].replace(0, np.nan)
)

previous["CREDIT_ANNUITY_RATIO"] = (
    previous["AMT_CREDIT"]
    / previous["AMT_ANNUITY"].replace(0, np.nan)
)

previous["APPROVED"] = (
    previous["NAME_CONTRACT_STATUS"] == "Approved"
).astype(int)

previous["REFUSED"] = (
    previous["NAME_CONTRACT_STATUS"] == "Refused"
).astype(int)

previous["CONTRACT_TERMINATED"] = (
    previous["DAYS_TERMINATION"].notna()
    & (previous["DAYS_TERMINATION"] <= 0)
).astype(int)

schedule_known = (
    previous["DAYS_LAST_DUE"].notna()
    & previous["DAYS_LAST_DUE_1ST_VERSION"].notna()
)

previous["SCHEDULE_CHANGED"] = (
    previous["DAYS_LAST_DUE"]
    != previous["DAYS_LAST_DUE_1ST_VERSION"]
).where(schedule_known).astype(float)

previous["SCHEDULE_SHIFT_DAYS"] = (
    previous["DAYS_LAST_DUE"]
    - previous["DAYS_LAST_DUE_1ST_VERSION"]
)

previous = previous.sort_values(
    ["SK_ID_CURR", "DAYS_DECISION"]
)

previous["APPLICATION_INTERVAL_DAYS"] = (
    previous.groupby("SK_ID_CURR")["DAYS_DECISION"]
    .diff()
)

In [17]:
previous_features = previous.groupby("SK_ID_CURR").agg(

    PREV_APPLICATION_COUNT=("SK_ID_PREV", "count"),

    PREV_MEAN_APPLICATION=("AMT_APPLICATION", "mean"),
    PREV_MAX_APPLICATION=("AMT_APPLICATION", "max"),

    PREV_MEAN_CREDIT=("AMT_CREDIT", "mean"),
    PREV_MAX_CREDIT=("AMT_CREDIT", "max"),

    PREV_MEAN_APPLICATION_CREDIT_RATIO=(
        "APPLICATION_CREDIT_RATIO", "mean"
    ),
    PREV_MEAN_APPLICATION_CREDIT_DIFF=(
        "APPLICATION_CREDIT_DIFF", "mean"
    ),

    PREV_MEAN_DOWN_PAYMENT_RATIO=(
        "DOWN_PAYMENT_RATIO", "mean"
    ),

    PREV_LAST_APPLICATION_DAYS=("DAYS_DECISION", "max"),
    PREV_MEAN_APPLICATION_DAYS=("DAYS_DECISION", "mean"),

    PREV_APPROVED_COUNT=("APPROVED", "sum"),
    PREV_REFUSED_COUNT=("REFUSED", "sum"),
    PREV_APPROVED_RATE=("APPROVED", "mean"),
    PREV_REFUSED_RATE=("REFUSED", "mean"),

    PREV_MEAN_CREDIT_ANNUITY_RATIO=(
        "CREDIT_ANNUITY_RATIO", "mean"
    ),
    PREV_STD_CREDIT_ANNUITY_RATIO=(
        "CREDIT_ANNUITY_RATIO", "std"
    ),

    PREV_MEAN_APPLICATION_INTERVAL=(
        "APPLICATION_INTERVAL_DAYS", "mean"
    ),
    PREV_MIN_APPLICATION_INTERVAL=(
        "APPLICATION_INTERVAL_DAYS", "min"
    ),

    PREV_TERMINATED_COUNT=("CONTRACT_TERMINATED", "sum"),

    PREV_SCHEDULE_CHANGED_COUNT=("SCHEDULE_CHANGED", "sum"),
    PREV_SCHEDULE_CHANGED_RATE=("SCHEDULE_CHANGED", "mean"),

    PREV_MEAN_SCHEDULE_SHIFT=("SCHEDULE_SHIFT_DAYS", "mean"),
    PREV_MAX_SCHEDULE_SHIFT=("SCHEDULE_SHIFT_DAYS", "max"),

).reset_index()

In [18]:
approved = previous[
    previous["NAME_CONTRACT_STATUS"] == "Approved"
]

approved_features = approved.groupby("SK_ID_CURR").agg(

    APPROVED_MEAN_CREDIT=("AMT_CREDIT", "mean"),
    APPROVED_MAX_CREDIT=("AMT_CREDIT", "max"),
    APPROVED_MEAN_ANNUITY=("AMT_ANNUITY", "mean"),
    APPROVED_MEAN_DOWN_PAYMENT=("AMT_DOWN_PAYMENT", "mean"),
    APPROVED_MEAN_CREDIT_RATIO=(
        "APPLICATION_CREDIT_RATIO", "mean"
    ),

    APPROVED_MEAN_CREDIT_ANNUITY_RATIO=(
        "CREDIT_ANNUITY_RATIO", "mean"
    ),
    APPROVED_STD_CREDIT_ANNUITY_RATIO=(
        "CREDIT_ANNUITY_RATIO", "std"
    ),

    APPROVED_INSURED_RATE=(
        "NFLAG_INSURED_ON_APPROVAL", "mean"
    ),

).reset_index()

In [19]:
contract_features = pd.crosstab(
    previous["SK_ID_CURR"],
    previous["NAME_CONTRACT_TYPE"]
).add_prefix("PREV_CONTRACT_COUNT_").reset_index()

In [20]:
refused = previous[
    previous["NAME_CONTRACT_STATUS"] == "Refused"
]

reject_features = pd.crosstab(
    refused["SK_ID_CURR"],
    refused["CODE_REJECT_REASON"]
).add_prefix("PREV_REJECT_REASON_").reset_index()


purpose_features = pd.crosstab(
    previous["SK_ID_CURR"],
    previous["NAME_CASH_LOAN_PURPOSE"]
).add_prefix("PREV_LOAN_PURPOSE_").reset_index()

In [21]:
recent_90 = previous[
    previous["DAYS_DECISION"].between(-90, 0)
]

recent_90_features = recent_90.groupby("SK_ID_CURR").agg(

    PREV_APPLICATION_COUNT_90D=("SK_ID_PREV", "count"),
    PREV_APPROVED_COUNT_90D=("APPROVED", "sum"),
    PREV_REFUSED_COUNT_90D=("REFUSED", "sum"),
    PREV_REFUSED_RATE_90D=("REFUSED", "mean"),

).reset_index()


recent_180 = previous[
    previous["DAYS_DECISION"].between(-180, 0)
]

recent_180_features = recent_180.groupby("SK_ID_CURR").agg(

    PREV_APPLICATION_COUNT_180D=("SK_ID_PREV", "count"),
    PREV_APPROVED_COUNT_180D=("APPROVED", "sum"),
    PREV_REFUSED_COUNT_180D=("REFUSED", "sum"),
    PREV_REFUSED_RATE_180D=("REFUSED", "mean"),

).reset_index()


recent_365 = previous[
    previous["DAYS_DECISION"].between(-365, 0)
]

recent_365_features = recent_365.groupby("SK_ID_CURR").agg(

    RECENT_APPLICATION_COUNT=("SK_ID_PREV", "count"),
    RECENT_APPROVED_COUNT=("APPROVED", "sum"),
    RECENT_REFUSED_COUNT=("REFUSED", "sum"),
    RECENT_REFUSED_RATE=("REFUSED", "mean"),

).reset_index()

In [22]:
new_feature_tables = [
    approved_features,
    contract_features,
    reject_features,
    purpose_features,
    recent_90_features,
    recent_180_features,
    recent_365_features
]

for features in new_feature_tables:
    previous_features = previous_features.merge(
        features,
        on="SK_ID_CURR",
        how="left",
        validate="one_to_one"
    )

count_cols = [
    col for col in previous_features.columns
    if (
        "COUNT" in col
        or col.startswith("PREV_REJECT_REASON_")
        or col.startswith("PREV_LOAN_PURPOSE_")
    )
]

previous_features[count_cols] = (
    previous_features[count_cols].fillna(0)
)

In [23]:
print("Размер:", previous_features.shape)
print("Дубликаты ID:", previous_features["SK_ID_CURR"].duplicated().sum())
numeric = previous_features.select_dtypes(include="number")
print("Количество inf:", np.isinf(numeric.to_numpy()).sum())

Размер: (338857, 81)
Дубликаты ID: 0
Количество inf: 0


## Pos features

Таблица POS_CASH_balance содержит помесячную информацию о состоянии предыдущих кредитов.

На её основе рассчитываются:

- Количество кредитов и исторических записей.

- Количество и доля месяцев с просрочками.

- Средняя и максимальная продолжительность просрочки.

- Доля месяцев с просрочками от 7 и 30 дней.

- Количество оставшихся платежей.

- Доля записей со статусами действующего и завершённого договора.

Отдельно рассчитываются показатели за последние 12 месяцев, чтобы учесть недавнюю платёжную дисциплину.

Полученные признаки агрегируются по SK_ID_CURR.

In [24]:
pos["POS_LATE"] = (pos["SK_DPD"] > 0).astype(int)
pos["POS_DPD_7"] = (pos["SK_DPD"] >= 7).astype(int)
pos["POS_DPD_30"] = (pos["SK_DPD"] >= 30).astype(int)

In [25]:
pos_features = pos.groupby("SK_ID_CURR").agg(

    POS_MONTH_COUNT=("SK_ID_PREV", "count"),
    POS_CREDIT_COUNT=("SK_ID_PREV", "nunique"),


    POS_LATE_COUNT=("POS_LATE", "sum"),
    POS_LATE_RATE=("POS_LATE", "mean"),

    POS_MAX_DPD=("SK_DPD", "max"),
    POS_MEAN_DPD=("SK_DPD", "mean"),

    POS_MAX_DPD_DEF=("SK_DPD_DEF", "max"),
    POS_MEAN_DPD_DEF=("SK_DPD_DEF", "mean"),

    POS_DPD_7_RATE=("POS_DPD_7", "mean"),
    POS_DPD_30_RATE=("POS_DPD_30", "mean"),

    POS_MEAN_FUTURE_INSTALMENTS=("CNT_INSTALMENT_FUTURE", "mean"),
    POS_MAX_FUTURE_INSTALMENTS=("CNT_INSTALMENT_FUTURE", "max"),
).reset_index()

In [26]:
recent_pos = pos[
    pos["MONTHS_BALANCE"].between(-12, -1)
]

recent_pos_features = recent_pos.groupby("SK_ID_CURR").agg(
    POS_RECENT_MONTH_COUNT=("SK_ID_PREV", "count"),

    POS_RECENT_LATE_RATE=("POS_LATE", "mean"),

    POS_RECENT_MAX_DPD=("SK_DPD", "max"),

    POS_RECENT_MEAN_DPD=("SK_DPD", "mean"),

    POS_RECENT_DPD_30_RATE=("POS_DPD_30", "mean"),
).reset_index()

pos_features = pos_features.merge(
    recent_pos_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

In [27]:
pos["POS_ACTIVE"] = (
    pos["NAME_CONTRACT_STATUS"] == "Active"
).astype(int)

pos["POS_COMPLETED"] = (
    pos["NAME_CONTRACT_STATUS"] == "Completed"
).astype(int)

In [28]:
status_features = pos.groupby("SK_ID_CURR").agg(
    POS_ACTIVE_RATE=("POS_ACTIVE", "mean"),
    POS_COMPLETED_RATE=("POS_COMPLETED", "mean"),
).reset_index()

pos_features = pos_features.merge(
    status_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

In [29]:
print("Размер:", pos_features.shape)
print("Дубликаты ID:", pos_features["SK_ID_CURR"].duplicated().sum())
numeric = pos_features.select_dtypes(include="number")
print("Количество inf:", np.isinf(numeric.to_numpy()).sum())

Размер: (337252, 20)
Дубликаты ID: 0
Количество inf: 0


## Credit card balance features

Таблица credit_card_balance содержит помесячную историю использования кредитных карт.

На её основе создаются признаки, характеризующие долговую нагрузку и платёжную дисциплину:

- Количество кредитных карт и исторических записей.

- Средний и максимальный баланс.

- Размер кредитного лимита.

- Отношение задолженности к кредитному лимиту (CC_LIMIT_USAGE).

- Отношения платежей к задолженности и минимальному обязательному платежу.

- Частота и продолжительность просрочек.

- Интенсивность использования кредитного лимита.

Дополнительно рассчитываются показатели за последние 12 месяцев.

Отдельный признак CC_HIGH_USAGE отмечает записи, в которых использование кредитного лимита превышает 80%.

После агрегации каждому клиенту соответствует одна строка.

In [30]:
cc["CC_LIMIT_USAGE"] = (
    cc["AMT_BALANCE"]
    / cc["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
)

cc["CC_PAYMENT_BALANCE_RATIO"] = (
    cc["AMT_PAYMENT_TOTAL_CURRENT"]
    / cc["AMT_BALANCE"].replace(0, np.nan)
)

cc["CC_PAYMENT_MIN_RATIO"] = (
    cc["AMT_PAYMENT_TOTAL_CURRENT"]
    / cc["AMT_INST_MIN_REGULARITY"].replace(0, np.nan)
)

cc["CC_LATE"] = (cc["SK_DPD"] > 0).astype(int)

cc["CC_DPD_7"] = (cc["SK_DPD"] >= 7).astype(int)

cc["CC_DPD_30"] = (cc["SK_DPD"] >= 30).astype(int)

In [31]:
cc_features = cc.groupby("SK_ID_CURR").agg(
    
    CC_CREDIT_COUNT=("SK_ID_PREV", "nunique"),
    CC_MONTH_COUNT=("SK_ID_PREV", "count"),

    CC_MEAN_BALANCE=("AMT_BALANCE", "mean"),
    CC_MAX_BALANCE=("AMT_BALANCE", "max"),


    CC_MEAN_LIMIT=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
    CC_MAX_LIMIT=("AMT_CREDIT_LIMIT_ACTUAL", "max"),

    CC_MEAN_LIMIT_USAGE=("CC_LIMIT_USAGE", "mean"),
    CC_MAX_LIMIT_USAGE=("CC_LIMIT_USAGE", "max"),

    CC_MEAN_PAYMENT=("AMT_PAYMENT_TOTAL_CURRENT", "mean"),
    CC_MAX_PAYMENT=("AMT_PAYMENT_TOTAL_CURRENT", "max"),

    CC_MEAN_PAYMENT_BALANCE_RATIO=("CC_PAYMENT_BALANCE_RATIO", "mean"),
    CC_MEAN_PAYMENT_MIN_RATIO=("CC_PAYMENT_MIN_RATIO", "mean"),

    CC_MEAN_DRAWINGS=("AMT_DRAWINGS_CURRENT", "mean"),
    CC_MAX_DRAWINGS=("AMT_DRAWINGS_CURRENT", "max"),

    CC_LATE_RATE=("CC_LATE", "mean"),
    CC_MAX_DPD=("SK_DPD", "max"),
    CC_MEAN_DPD=("SK_DPD", "mean"),

    CC_MAX_DPD_DEF=("SK_DPD_DEF", "max"),

    CC_DPD_7_RATE=("CC_DPD_7", "mean"),
    CC_DPD_30_RATE=("CC_DPD_30", "mean"),
).reset_index()

In [32]:
recent_cc = cc[
    cc["MONTHS_BALANCE"].between(-12, -1)
]

recent_cc_features = recent_cc.groupby("SK_ID_CURR").agg(
    CC_RECENT_MONTH_COUNT=("SK_ID_PREV", "count"),

    CC_RECENT_MEAN_BALANCE=("AMT_BALANCE", "mean"),

    CC_RECENT_MEAN_LIMIT_USAGE=("CC_LIMIT_USAGE", "mean"),

    CC_RECENT_MAX_LIMIT_USAGE=("CC_LIMIT_USAGE", "max"),

    CC_RECENT_MEAN_PAYMENT=("AMT_PAYMENT_TOTAL_CURRENT", "mean"),

    CC_RECENT_LATE_RATE=("CC_LATE", "mean"),

    CC_RECENT_MAX_DPD=("SK_DPD", "max"),
).reset_index()

cc_features = cc_features.merge(
    recent_cc_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

In [33]:
cc["CC_HIGH_USAGE"] = (
    cc["CC_LIMIT_USAGE"] > 0.8
).astype(int)

usage_features = cc.groupby("SK_ID_CURR").agg(
    CC_HIGH_USAGE_RATE=("CC_HIGH_USAGE", "mean")
).reset_index()

cc_features = cc_features.merge(
    usage_features,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

In [34]:
print("Размер:", cc_features.shape)
print("Дубликаты ID:", cc_features["SK_ID_CURR"].duplicated().sum())
numeric = cc_features.select_dtypes(include="number")
print("Количество inf:", np.isinf(numeric.to_numpy()).sum())

Размер: (103558, 29)
Дубликаты ID: 0
Количество inf: 0


### bureau balance

Был проведён эксперимент по добавлению агрегированных признаков из таблицы bureau_balance.

Результаты 5-fold кросс-валидации:

С bureau_balance: ROC-AUC = 0.790647.

Без bureau_balance: ROC-AUC = 0.790769.

Добавление признаков не привело к улучшению качества модели. Разница между результатами незначительна.

Поскольку таблица содержит около 27,3 млн строк и требует дополнительных вычислительных ресурсов, было принято решение исключить её из финального набора признаков.

## Соединение таблиц

Агрегированные признаки из дополнительных таблиц объединяются с основной таблицей по SK_ID_CURR.

Используется left join, чтобы сохранить всех клиентов из исходных application_train и application_test, включая тех, для кого отсутствует дополнительная кредитная история.

Параметр validate="one_to_one" позволяет проверить уникальность ключей объединения и предотвратить случайное размножение строк.

После объединения рассчитываются дополнительные признаки кредитной нагрузки, объединяющие данные из разных источников, например отношение общей задолженности по кредитному бюро к доходу заёмщика.

Подготовка обучающей и тестовой выборок выполняется одинаковым способом.

In [35]:
feature_tables = [
    bureau_features,
    installments_features,
    previous_features,
    pos_features,
    cc_features,
]

def merge_features(df, feature_tables):
    df = df.copy()

    for features in feature_tables:
        df = df.merge(
            features,
            on="SK_ID_CURR",
            how="left",
            validate="one_to_one"
        )
    df["BUREAU_DEBT_TO_INCOME"] = (
        df["TOTAL_DEBT"]
        / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )
    
    df["BUREAU_ACTIVE_DEBT_TO_INCOME"] = (
        df["ACTIVE_TOTAL_DEBT"]
        / df["AMT_INCOME_TOTAL"].replace(0, np.nan)
    )
    
    df["BUREAU_DEBT_CREDIT_RATIO"] = (
        df["TOTAL_DEBT"]
        / df["TOTAL_CREDIT"].replace(0, np.nan)
    )
    
    df["ACTIVE_CREDIT_RATE"] = (
        df["ACTIVE_CREDIT_COUNT"]
        / df["CREDIT_COUNT"].replace(0, np.nan))

    return df

In [36]:
train = application_features(application_train)
test = application_features(application_test)
application_columns = train.columns
pd.Series(application_columns).to_csv("application_columns.csv", index=False)

In [37]:
train = merge_features(train, feature_tables)
test = merge_features(test, feature_tables)

In [38]:
train.to_csv("features_train.csv")
test.to_csv("features_test.csv")

## Проверка данных

После объединения таблиц проверяются:

- Сохранение исходного количества клиентов и их идентификаторов.

- Отсутствие дублирующихся SK_ID_CURR.

- Совпадение наборов признаков между обучающей и тестовой выборками.

- Доли пропущенных значений.

- Наличие бесконечных значений.

- Отсутствие повторяющихся названий столбцов.

In [39]:
assert len(train) == len(application_train)
assert len(test) == len(application_test)

assert train["SK_ID_CURR"].equals(application_train["SK_ID_CURR"])
assert test["SK_ID_CURR"].equals(application_test["SK_ID_CURR"])

print("Train:", train.shape)
print("Test:", test.shape)

Train: (307511, 332)
Test: (48744, 331)


In [40]:
print("Только в train:", set(train.columns) - set(test.columns))
print("Только в test:", set(test.columns) - set(train.columns))

Только в train: {'TARGET'}
Только в test: set()


In [41]:
print("Дубликаты клиентов train:", train["SK_ID_CURR"].duplicated().sum())
print("Дубликаты клиентов test:", test["SK_ID_CURR"].duplicated().sum())

Дубликаты клиентов train: 0
Дубликаты клиентов test: 0


In [42]:
missing = train.isna().mean().sort_values(ascending=False)

missing.head(20)

PREV_REFUSED_RATE_90D            0.853202
CC_MEAN_PAYMENT_MIN_RATIO        0.807230
CC_MEAN_PAYMENT_BALANCE_RATIO    0.806183
CC_RECENT_MEAN_LIMIT_USAGE       0.768070
CC_RECENT_MAX_LIMIT_USAGE        0.768070
CC_MAX_LIMIT_USAGE               0.720218
CC_MEAN_LIMIT_USAGE              0.720218
CC_MEAN_DPD                      0.717392
CC_MAX_DPD_DEF                   0.717392
CC_MAX_DPD                       0.717392
CC_LATE_RATE                     0.717392
CC_MAX_DRAWINGS                  0.717392
CC_MEAN_DRAWINGS                 0.717392
CC_RECENT_MEAN_PAYMENT           0.717392
CC_RECENT_LATE_RATE              0.717392
CC_CREDIT_COUNT                  0.717392
CC_MONTH_COUNT                   0.717392
CC_RECENT_MAX_DPD                0.717392
CC_HIGH_USAGE_RATE               0.717392
CC_MAX_BALANCE                   0.717392
dtype: float64

In [43]:
missing = test.isna().mean().sort_values(ascending=False)

missing.head(20)

PREV_REFUSED_RATE_90D            0.870589
CC_MEAN_PAYMENT_MIN_RATIO        0.782189
CC_MEAN_PAYMENT_BALANCE_RATIO    0.781860
PREV_REFUSED_RATE_180D           0.746348
CC_RECENT_MEAN_LIMIT_USAGE       0.721812
CC_RECENT_MAX_LIMIT_USAGE        0.721812
CAR_TO_EMPLOYED_RATIO            0.697583
COMMONAREA_AVG                   0.687161
COMMONAREA_MEDI                  0.687161
COMMONAREA_MODE                  0.687161
NONLIVINGAPARTMENTS_AVG          0.684125
NONLIVINGAPARTMENTS_MODE         0.684125
NONLIVINGAPARTMENTS_MEDI         0.684125
FONDKAPREMONT_MODE               0.672842
LIVINGAPARTMENTS_MEDI            0.672493
LIVINGAPARTMENTS_AVG             0.672493
LIVINGAPARTMENTS_MODE            0.672493
FLOORSMIN_MEDI                   0.666051
FLOORSMIN_AVG                    0.666051
FLOORSMIN_MODE                   0.666051
dtype: float64

In [44]:
numeric = train.select_dtypes(include="number")

inf_counts = np.isinf(numeric).sum()
inf_counts[inf_counts > 0]

Series([], dtype: int64)

In [45]:
print("Повторяющиеся столбцы:",
      test.columns[test.columns.duplicated()].tolist())

Повторяющиеся столбцы: []


Проверка потенциальных временных утечек

Для снижения риска использования информации из будущего проверяются временные характеристики дополнительных таблиц.

In [46]:
print("Future scheduled payments:",
      (installments["DAYS_INSTALMENT"] > 0).sum())

print("Future actual payments:",
      (installments["DAYS_ENTRY_PAYMENT"] > 0).sum())

Future scheduled payments: 0
Future actual payments: 0


In [47]:
print(pos["MONTHS_BALANCE"].max())
print(cc["MONTHS_BALANCE"].max())

-1
-1


In [48]:
cols = [
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION"
]

for col in cols:
    valid = previous[col].replace(365243, np.nan)

    print(f"{col}:")
    print("Будущих дат:", (valid > 0).sum())
    print("Пропусков:", valid.isna().sum())
    print()

DAYS_LAST_DUE_1ST_VERSION:
Будущих дат: 224392
Пропусков: 766929

DAYS_LAST_DUE:
Будущих дат: 0
Пропусков: 884286

DAYS_TERMINATION:
Будущих дат: 0
Пропусков: 898978



In [49]:
mask = (
    (previous["DAYS_LAST_DUE_1ST_VERSION"] > 0)
    & (previous["DAYS_LAST_DUE_1ST_VERSION"] != 365243)
    & (previous["DAYS_TERMINATION"] <= 0)
)

print("Количество записей:", mask.sum())

Количество записей: 83390


In [50]:
original = previous["DAYS_LAST_DUE_1ST_VERSION"].replace(365243, np.nan)
updated = previous["DAYS_LAST_DUE"].replace(365243, np.nan)

mask = original.notna() & updated.notna()

print("Записей с двумя известными датами:", mask.sum())
print("Изменённый график:", (original[mask] != updated[mask]).sum())

print(
    "Доля изменений:",
    (original[mask] != updated[mask]).mean()
)

Записей с двумя известными датами: 761917
Изменённый график: 371116
Доля изменений: 0.48708192624655966


In [51]:
mask = (
    original.notna()
    & updated.notna()
    & original.ne(updated)
    & (
        previous["DAYS_TERMINATION"].isna()
        | previous["DAYS_TERMINATION"].eq(365243)
    )
)

print("Изменения графика без известной даты закрытия:", mask.sum())

Изменения графика без известной даты закрытия: 1378


Проверки показали, что объединение не изменило количество клиентов: обучающая выборка содержит 307 511 наблюдений, тестовая — 48 744.

Наборы столбцов совпадают, за исключением TARGET, который присутствует только в обучающей выборке.

Наибольшая доля пропусков наблюдается среди признаков недавних предыдущих заявок и кредитных карт. Это может объясняться отсутствием соответствующей истории у части клиентов.

Такие пропуски не удаляются автоматически, поскольку их наличие может содержать информацию о заёмщике.


В installments_payments не обнаружено плановых или фактических дат платежей позже даты текущей заявки.

В POS_CASH_balance и credit_card_balance максимальное значение MONTHS_BALANCE равно −1, что соответствует последнему месяцу перед текущей заявкой.

Для previous_application отдельно исследованы первоначальные и изменённые плановые даты погашения кредитов.

Проверки позволяют выявить потенциально проблемные признаки, однако не гарантируют полного отсутствия временных утечек, поскольку точное время возникновения некоторых событий невозможно установить по имеющимся данным.

## Вывод


В ходе Feature Engineering были созданы дополнительные признаки на основе характеристик заявителей, кредитной истории, предыдущих заявок и платёжной дисциплины.

Исторические данные агрегированы до уровня клиента и объединены с основной таблицей. При объединении сохранены все исходные наблюдения, а наборы признаков обучающей и тестовой выборок согласованы.

Проведены проверки уникальности идентификаторов, пропусков, бесконечных значений и потенциальных временных утечек.

В следующем этапе будет оценено влияние дополнительных признаков на качество CatBoost с помощью стратифицированной кросс-валидации.